In [1]:
#########################################################################
# For poor windows users, uncomment the following lines to share the magic of RLang with Python

import os

# Optional: also make sure R’s own locale is UTF‑8 if possible
os.environ['LANG'] = 'en_US.UTF-8'

# Use the location of your R installation, you'd better add the R bin directory to your PATH first
os.environ['R_HOME'] = 'D:\\R-4.3.3\\'

#########################################################################

import rpy2
import pandas as pd

%load_ext rpy2.ipython

%R require("ggplot2")
%R require("data.table")
%R require("lme4") # for lmer() linear model with random effects
%R require("ordinal")

f:\Study\CS\miniconda\envs\py310\lib\site-packages\rpy2\robjects\packages.py:367: UserWarning: The symbol 'quartz' is not in this R namespace/package.
  warnings.warn(


Loading required package: ggplot2
Need help? Try Stackoverflow: https://stackoverflow.com/tags/ggplot2


Loading required package: data.table
data.table 1.15.2 using 6 threads (see ?getDTthreads).  Latest news: r-datatable.com


Loading required package: lme4
Loading required package: Matrix


Loading required package: ordinal


In [2]:
df = pd.read_csv("metrics_collection.csv")
df.head()

,id,conversation,question_id,model_name,content,time,time3,mt,exp,nll,...,word_speed,line_speed,pixel_speed,saccade,fft,rrt,tft,skip_rate,fixation_time,subject_id
0,1,1,100,vicuna-13b-v1.2,I do not actually have subjective feelings or ...,28163,16827,2.0,1.75,152.017463,...,331.329412,4693.833333,2.936585,143,282.2,232.4,282.2,0.327273,15850.000000,1
1,1,2,100,claude-v1,"As an AI language model, I do not have the abi...",17616,16827,1.0,1.25,176.372082,...,320.290909,3523.200000,1.288177,79,348.6,448.2,448.2,0.435294,25783.333333,1
2,2,1,88,gpt-3.5-turbo,"As soon as the alarm clock went off, Jane knew...",28752,5967,2.0,2.00,152.409554,...,315.956044,4107.428571,1.821815,141,116.2,232.4,348.6,0.494505,26450.000000,1
3,2,2,88,vicuna-13b-v1.2,"It was a strange and unsettling feeling, wakin...",23949,5967,1.0,1.00,126.195426,...,342.128571,3991.500000,1.962668,130,348.6,116.2,199.2,0.428571,21616.666667,1
4,3,1,100,gpt-3.5-turbo,I do not actually have subjective feelings or ...,22202,22419,2.0,2.00,127.155161,...,267.493976,3171.714286,2.315026,105,265.6,1228.4,282.2,0.272727,23050.000000,1


### Fit $M_0$ with nll as the only predictor

In [3]:
%%R -i df

# LM
dt <- data.table(df)

m0 <- lm(mt ~ nll + model_name, data = dt)
cat("R-squared:", summary(m0)$r.squared, "\n")
cat("Adjusted R-squared:", summary(m0)$adj.r.squared, "\n")
summary(m0)

R-squared: 0.1997929 
Adjusted R-squared: 0.1948279 

Call:
lm(formula = mt ~ nll + model_name, data = dt)

Residuals:
     Min       1Q   Median       3Q      Max 
-1.03373 -0.40653 -0.03373  0.37421  0.95205 

Coefficients:
                            Estimate Std. Error t value Pr(>|t|)    
(Intercept)                0.7014744  0.0645935  10.860  < 2e-16 ***
nll                        0.0045975  0.0004135  11.117  < 2e-16 ***
model_nameclaude-v1       -0.1058150  0.0537412  -1.969   0.0492 *  
model_namegpt-3.5-turbo    0.3472725  0.0402541   8.627  < 2e-16 ***
model_namegpt-4            0.0807241  0.0550066   1.468   0.1426    
model_namellama-13b       -0.2707434  0.0607307  -4.458 9.24e-06 ***
model_namevicuna-13b-v1.2  0.2153429  0.0462084   4.660 3.60e-06 ***
---
Signif. codes:  0 '***' 0.001 '**' 0.01 '*' 0.05 '.' 0.1 ' ' 1

Residual standard error: 0.4188 on 967 degrees of freedom
Multiple R-squared:  0.1998,	Adjusted R-squared:  0.1948 
F-statistic: 40.24 on 6 and 967 DF,  p

In [4]:
%%R -i df

# GLMER
dt <- data.table(df)

dt$mt = (dt$mt - min(dt$mt)) / (max(dt$mt) - min(dt$mt))
table(dt$mt)
#   0 0.5   1
# 424 126 424

# ignore the 0.5 group
dt <- dt[mt != 0.5]

m0 <- glmer(mt ~ nll + (1 | model_name), data = dt, family = binomial(link = "logit"))
# summary(m0)
print("AIC:")
print(AIC(m0))
print("BIC:")
print(BIC(m0))

summary_m0 <- summary(m0)

# Extract fixed effects coefficients table
coef_table <- summary_m0$coefficients

# coef_table is a matrix with columns: Estimate, Std. Error, z value, Pr(>|z|)
# For example, to get beta_0 (Intercept)
beta_0 <- coef_table["(Intercept)", "Estimate"]
se_beta_0 <- coef_table["(Intercept)", "Std. Error"]
z_beta_0 <- coef_table["(Intercept)", "z value"]

print(paste("beta_0:", beta_0))
print(paste("se_beta_0:", se_beta_0))
print(paste("z_beta_0:", z_beta_0))

# Similarly for nll
beta_nll <- coef_table["nll", "Estimate"]
se_beta_nll <- coef_table["nll", "Std. Error"]
z_beta_nll <- coef_table["nll", "z value"]

print(paste("beta_nll:", beta_nll))
print(paste("se_beta_nll:", se_beta_nll))
print(paste("z_beta_nll:", z_beta_nll))

[1] "AIC:"
[1] 1003.006
[1] "BIC:"
[1] 1017.235
[1] "beta_0: -4.36560852358185"
[1] "se_beta_0: 0.665923707087516"
[1] "z_beta_0: -6.55571873642294"
[1] "beta_nll: 0.0269907324993069"
[1] "se_beta_nll: 0.00317390619967131"
[1] "z_beta_nll: 8.50394775437979"


In [5]:
%%R -i df
dt <- data.table(df)

# Regular model with no random intercepts
m0_reg <- lm(mt ~ nll, data = dt)

# print R-squared
print("R-squared: ")
print(summary(m0_reg)$r.squared)
print("Adjusted R-squared: ")
print(summary(m0_reg)$adj.r.squared)

# summary(m0_reg)

[1] "R-squared: "
[1] 0.06181761
[1] "Adjusted R-squared: "
[1] 0.0608524


### M_2: Fixation time (FT)

In [6]:
%%R -i df
dt <- data.table(df)

dt$mt = (dt$mt - min(dt$mt)) / (max(dt$mt) - min(dt$mt))
table(dt$mt)
#   0 0.5   1
# 424 126 424

# ignore the 0.5 group
dt <- dt[mt != 0.5]

m2_reg <- glmer(mt ~ log(fixation_time) + (1 | model_name) + (1 | subject_id), data = dt, family = binomial(link = "logit"))
# summary(m2_reg)

print("AIC:")
print(AIC(m2_reg))
print("BIC:")
print(BIC(m2_reg))

# Extract fixed effects coefficients table
coef_table <- summary(m2_reg)$coefficients
# coef_table is a matrix with columns: Estimate, Std. Error, z value, Pr(>|z|)
# For example, to get beta_0 (Intercept)
beta_0 <- coef_table["(Intercept)", "Estimate"]
se_beta_0 <- coef_table["(Intercept)", "Std. Error"]
z_beta_0 <- coef_table["(Intercept)", "z value"]

print(paste("beta_0:", beta_0))
print(paste("se_beta_0:", se_beta_0))
print(paste("z_beta_0:", z_beta_0))

# Similarly for log(fixation_time)
beta_log_fixation_time <- coef_table["log(fixation_time)", "Estimate"]
se_beta_log_fixation_time <- coef_table["log(fixation_time)", "Std. Error"]
z_beta_log_fixation_time <- coef_table["log(fixation_time)", "z value"]

print(paste("beta_log_fixation_time:", beta_log_fixation_time))
print(paste("se_beta_log_fixation_time:", se_beta_log_fixation_time))
print(paste("z_beta_log_fixation_time:", z_beta_log_fixation_time))

[1] "AIC:"
[1] 1093.822
[1] "BIC:"
[1] 1112.793
[1] "beta_0: 4.70150343825898"
[1] "se_beta_0: 1.48019375274797"
[1] "z_beta_0: 3.17627569332101"
[1] "beta_log_fixation_time: -0.518837562802682"
[1] "se_beta_log_fixation_time: 0.147657714452326"
[1] "z_beta_log_fixation_time: -3.51378568148025"


boundary (singular) fit: see help('isSingular')


In [7]:
%%R -i df
dt <- data.table(df)

m2_reg <- lm(mt ~ log(fixation_time), data = dt)
print("R-squared: ")
print(summary(m2_reg)$r.squared)
print("Adjusted R-squared: ")
print(summary(m2_reg)$adj.r.squared)
summary(m2_reg)

[1] "R-squared: "
[1] 0.01343493
[1] "Adjusted R-squared: "
[1] 0.01241995

Call:
lm(formula = mt ~ log(fixation_time), data = dt)

Residuals:
    Min      1Q  Median      3Q     Max 
-0.6874 -0.4794  0.0008  0.4911  0.5999 

Coefficients:
                   Estimate Std. Error t value Pr(>|t|)    
(Intercept)         2.51666    0.27983   8.993  < 2e-16 ***
log(fixation_time) -0.10472    0.02878  -3.638 0.000289 ***
---
Signif. codes:  0 '***' 0.001 '**' 0.01 '*' 0.05 '.' 0.1 ' ' 1

Residual standard error: 0.4639 on 972 degrees of freedom
Multiple R-squared:  0.01343,	Adjusted R-squared:  0.01242 
F-statistic: 13.24 on 1 and 972 DF,  p-value: 0.000289



### M_1: pixel_dwelling_time (PDT)

In [7]:
%%R -i df
dt <- data.table(df)

dt$mt = (dt$mt - min(dt$mt)) / (max(dt$mt) - min(dt$mt))
table(dt$mt)
#   0 0.5   1
# 424 126 424

# ignore the 0.5 group
dt <- dt[mt != 0.5]

m1 <- glmer(mt ~ log(pixel_speed) + (1|model_name) + (1|subject_id), data = dt, family = binomial(link = "logit"))

print("AIC:")
print(AIC(m1))
print("BIC:")
print(BIC(m1))
# summary(m1)

# Extract fixed effects coefficients table
coef_table <- summary(m1)$coefficients
# coef_table is a matrix with columns: Estimate, Std. Error, z value, Pr(>|z|)
# For example, to get beta_0 (Intercept)
beta_0 <- coef_table["(Intercept)", "Estimate"]
se_beta_0 <- coef_table["(Intercept)", "Std. Error"]
z_beta_0 <- coef_table["(Intercept)", "z value"]

print(paste("beta_0:", beta_0))
print(paste("se_beta_0:", se_beta_0))
print(paste("z_beta_0:", z_beta_0))

# Similarly for log(pixel_speed)
beta_log_pixel_speed <- coef_table["log(pixel_speed)", "Estimate"]
se_beta_log_pixel_speed <- coef_table["log(pixel_speed)", "Std. Error"]
z_beta_log_pixel_speed <- coef_table["log(pixel_speed)", "z value"]

print(paste("beta_log_pixel_speed:", beta_log_pixel_speed))
print(paste("se_beta_log_pixel_speed:", se_beta_log_pixel_speed))
print(paste("z_beta_log_pixel_speed:", z_beta_log_pixel_speed))

[1] "AIC:"
[1] 1067.124
[1] "BIC:"
[1] 1086.096
[1] "beta_0: -0.745632946462795"
[1] "se_beta_0: 0.338654785396467"
[1] "z_beta_0: -2.20174932886265"
[1] "beta_log_pixel_speed: 0.885007185159174"
[1] "se_beta_log_pixel_speed: 0.146662531306085"
[1] "z_beta_log_pixel_speed: 6.03431004004808"


boundary (singular) fit: see help('isSingular')


In [9]:
%%R -i df
dt <- data.table(df)

m1_reg <- lm(mt ~ log(pixel_speed), data = dt)
print("R-squared: ")
print(summary(m1_reg)$r.squared)
print("Adjusted R-squared: ")
print(summary(m1_reg)$adj.r.squared)
summary(m1_reg)

[1] "R-squared: "
[1] 0.064363
[1] "Adjusted R-squared: "
[1] 0.06340041

Call:
lm(formula = mt ~ log(pixel_speed), data = dt)

Residuals:
     Min       1Q   Median       3Q      Max 
-0.78953 -0.45767  0.00597  0.44835  0.82258 

Coefficients:
                 Estimate Std. Error t value Pr(>|t|)    
(Intercept)       1.39411    0.01942  71.781  < 2e-16 ***
log(pixel_speed)  0.21325    0.02608   8.177 9.03e-16 ***
---
Signif. codes:  0 '***' 0.001 '**' 0.01 '*' 0.05 '.' 0.1 ' ' 1

Residual standard error: 0.4517 on 972 degrees of freedom
Multiple R-squared:  0.06436,	Adjusted R-squared:  0.0634 
F-statistic: 66.86 on 1 and 972 DF,  p-value: 9.028e-16



### M_3: Backward Saccade Rate (BSR)

In [8]:
%%R -i df
dt <- data.table(df)

dt$mt = (dt$mt - min(dt$mt)) / (max(dt$mt) - min(dt$mt))
table(dt$mt)
#   0 0.5   1
# 424 126 424

# ignore the 0.5 group
dt <- dt[mt != 0.5]

m3 <- glmer(mt ~ N100 + (1|model_name) + (1|subject_id), data = dt, family = binomial(link = "logit"))

print("AIC:")
print(AIC(m3))
print("BIC:")
print(BIC(m3))
# summary(m3)

# Extract fixed effects coefficients table
coef_table <- summary(m3)$coefficients
# coef_table is a matrix with columns: Estimate, Std. Error, z value, Pr(>|z|)
# For example, to get beta_0 (Intercept)
beta_0 <- coef_table["(Intercept)", "Estimate"]
se_beta_0 <- coef_table["(Intercept)", "Std. Error"]
z_beta_0 <- coef_table["(Intercept)", "z value"]

print(paste("beta_0:", beta_0))
print(paste("se_beta_0:", se_beta_0))
print(paste("z_beta_0:", z_beta_0))

# Similarly for N100
beta_N100 <- coef_table["N100", "Estimate"]
se_beta_N100 <- coef_table["N100", "Std. Error"]
z_beta_N100 <- coef_table["N100", "z value"]

print(paste("beta_N100:", beta_N100))
print(paste("se_beta_N100:", se_beta_N100))
print(paste("z_beta_N100:", z_beta_N100))

[1] "AIC:"
[1] 1092.809
[1] "BIC:"
[1] 1111.78
[1] "beta_0: -1.04212556698916"
[1] "se_beta_0: 0.384673604730063"
[1] "z_beta_0: -2.70911638899802"
[1] "beta_N100: 0.015200220873892"
[1] "se_beta_N100: 0.00416329488864175"
[1] "z_beta_N100: 3.65100750258193"


boundary (singular) fit: see help('isSingular')


In [11]:
%%R -i df
dt <- data.table(df)

m3_reg <- lm(mt ~ N100, data = dt)
print("R-squared: ")
print(summary(m3_reg)$r.squared)
print("Adjusted R-squared: ")
print(summary(m3_reg)$adj.r.squared)
summary(m3_reg)

[1] "R-squared: "
[1] 0.0325483
[1] "Adjusted R-squared: "
[1] 0.03155298

Call:
lm(formula = mt ~ N100, data = dt)

Residuals:
     Min       1Q   Median       3Q      Max 
-0.98567 -0.46617  0.00221  0.47059  0.69194 

Coefficients:
            Estimate Std. Error t value Pr(>|t|)    
(Intercept) 1.280960   0.041034  31.217  < 2e-16 ***
N100        0.004517   0.000790   5.719 1.43e-08 ***
---
Signif. codes:  0 '***' 0.001 '**' 0.01 '*' 0.05 '.' 0.1 ' ' 1

Residual standard error: 0.4594 on 972 degrees of freedom
Multiple R-squared:  0.03255,	Adjusted R-squared:  0.03155 
F-statistic:  32.7 on 1 and 972 DF,  p-value: 1.43e-08



### M_comb: FT + PDT

In [12]:
%%R -i df
dt <- data.table(df)

dt$mt = (dt$mt - min(dt$mt)) / (max(dt$mt) - min(dt$mt))
table(dt$mt)
#   0 0.5   1
# 424 126 424

# ignore the 0.5 group
dt <- dt[mt != 0.5]

m_comb <- glmer(mt ~ log(fixation_time) + log(pixel_speed) + (1|model_name) + (1|subject_id), data = dt, family = binomial(link = "logit"))

print("AIC:")
print(AIC(m_comb))
print("BIC:")
print(BIC(m_comb))
summary(m_comb)

[1] "AIC:"
[1] 1042.374
[1] "BIC:"
[1] 1066.089
Generalized linear mixed model fit by maximum likelihood (Laplace
  Approximation) [glmerMod]
 Family: binomial  ( logit )
Formula: mt ~ log(fixation_time) + log(pixel_speed) + (1 | model_name) +  
    (1 | subject_id)
   Data: dt

      AIC       BIC    logLik -2*log(L)  df.resid 
   1042.4    1066.1    -516.2    1032.4       843 

Scaled residuals: 
     Min       1Q   Median       3Q      Max 
-2.66120 -0.82413 -0.00612  0.85902  3.01574 

Random effects:
 Groups     Name        Variance Std.Dev.
 subject_id (Intercept) 0.0000   0.0000  
 model_name (Intercept) 0.6124   0.7826  
Number of obs: 848, groups:  subject_id, 37; model_name, 6

Fixed effects:
                   Estimate Std. Error z value Pr(>|z|)    
(Intercept)          7.1594     1.6079   4.453 8.48e-06 ***
log(fixation_time)  -0.8242     0.1646  -5.008 5.50e-07 ***
log(pixel_speed)     1.1111     0.1630   6.816 9.38e-12 ***
---
Signif. codes:  0 '***' 0.001 '**' 0.01 '*' 

boundary (singular) fit: see help('isSingular')


In [13]:
%%R -i df
dt <- data.table(df)

m3_reg <- lm(mt ~ log(fixation_time) + log(pixel_speed), data = dt)

print("R-squared: ")
print(summary(m3_reg)$r.squared)
print("Adjusted R-squared: ")
print(summary(m3_reg)$adj.r.squared)
summary(m3_reg)

[1] "R-squared: "
[1] 0.09122022
[1] "Adjusted R-squared: "
[1] 0.08934838

Call:
lm(formula = mt ~ log(fixation_time) + log(pixel_speed), data = dt)

Residuals:
     Min       1Q   Median       3Q      Max 
-0.86944 -0.44866  0.01372  0.46981  0.83846 

Coefficients:
                   Estimate Std. Error t value Pr(>|t|)    
(Intercept)         2.84264    0.27108  10.486  < 2e-16 ***
log(fixation_time) -0.15048    0.02809  -5.357 1.06e-07 ***
log(pixel_speed)    0.23827    0.02614   9.117  < 2e-16 ***
---
Signif. codes:  0 '***' 0.001 '**' 0.01 '*' 0.05 '.' 0.1 ' ' 1

Residual standard error: 0.4454 on 971 degrees of freedom
Multiple R-squared:  0.09122,	Adjusted R-squared:  0.08935 
F-statistic: 48.73 on 2 and 971 DF,  p-value: < 2.2e-16



### M_comb: BSR + PDT

In [14]:
%%R -i df
dt <- data.table(df)

dt$mt = (dt$mt - min(dt$mt)) / (max(dt$mt) - min(dt$mt))
table(dt$mt)
#   0 0.5   1
# 424 126 424

# ignore the 0.5 group
dt <- dt[mt != 0.5]

m_comb <- glmer(mt ~ N100 + log(pixel_speed) + (1|model_name) + (1|subject_id), data = dt, family = binomial(link = "logit"))

print("AIC:")
print(AIC(m_comb))
print("BIC:")
print(BIC(m_comb))
summary(m_comb)

[1] "AIC:"
[1] 1068.841
[1] "BIC:"
[1] 1092.556
Generalized linear mixed model fit by maximum likelihood (Laplace
  Approximation) [glmerMod]
 Family: binomial  ( logit )
Formula: mt ~ N100 + log(pixel_speed) + (1 | model_name) + (1 | subject_id)
   Data: dt

      AIC       BIC    logLik -2*log(L)  df.resid 
   1068.8    1092.6    -529.4    1058.8       843 

Scaled residuals: 
    Min      1Q  Median      3Q     Max 
-2.0670 -0.8674  0.1024  0.7950  2.9340 

Random effects:
 Groups     Name        Variance  Std.Dev. 
 subject_id (Intercept) 1.668e-10 1.291e-05
 model_name (Intercept) 5.873e-01 7.664e-01
Number of obs: 848, groups:  subject_id, 37; model_name, 6

Fixed effects:
                  Estimate Std. Error z value Pr(>|z|)    
(Intercept)      -0.845082   0.383656  -2.203   0.0276 *  
N100              0.002577   0.004842   0.532   0.5946    
log(pixel_speed)  0.840011   0.169075   4.968 6.75e-07 ***
---
Signif. codes:  0 '***' 0.001 '**' 0.01 '*' 0.05 '.' 0.1 ' ' 1

Correlat

boundary (singular) fit: see help('isSingular')


In [15]:
%%R -i df
dt <- data.table(df)

m3_reg <- lm(mt ~ N100 + log(pixel_speed), data = dt)

print("R-squared: ")
print(summary(m3_reg)$r.squared)
print("Adjusted R-squared: ")
print(summary(m3_reg)$adj.r.squared)
summary(m3_reg)

[1] "R-squared: "
[1] 0.06665799
[1] "Adjusted R-squared: "
[1] 0.06473555

Call:
lm(formula = mt ~ N100 + log(pixel_speed), data = dt)

Residuals:
     Min       1Q   Median       3Q      Max 
-0.80684 -0.45436  0.01453  0.45214  0.83174 

Coefficients:
                  Estimate Std. Error t value Pr(>|t|)    
(Intercept)      1.3375550  0.0414292  32.285  < 2e-16 ***
N100             0.0014407  0.0009324   1.545    0.123    
log(pixel_speed) 0.1864594  0.0313008   5.957 3.59e-09 ***
---
Signif. codes:  0 '***' 0.001 '**' 0.01 '*' 0.05 '.' 0.1 ' ' 1

Residual standard error: 0.4514 on 971 degrees of freedom
Multiple R-squared:  0.06666,	Adjusted R-squared:  0.06474 
F-statistic: 34.67 on 2 and 971 DF,  p-value: 2.85e-15



### M_comb: BSR + FT

In [16]:
%%R -i df
dt <- data.table(df)

dt$mt = (dt$mt - min(dt$mt)) / (max(dt$mt) - min(dt$mt))
table(dt$mt)
#   0 0.5   1
# 424 126 424

# ignore the 0.5 group
dt <- dt[mt != 0.5]

m_comb <- glmer(mt ~ N100 + log(fixation_time) + (1|model_name) + (1|subject_id), data = dt, family = binomial(link = "logit"))

print("AIC:")
print(AIC(m_comb))
print("BIC:")
print(BIC(m_comb))
summary(m_comb)

[1] "AIC:"
[1] 1069.214
[1] "BIC:"
[1] 1092.929
Generalized linear mixed model fit by maximum likelihood (Laplace
  Approximation) [glmerMod]
 Family: binomial  ( logit )
Formula: mt ~ N100 + log(fixation_time) + (1 | model_name) + (1 | subject_id)
   Data: dt

      AIC       BIC    logLik -2*log(L)  df.resid 
   1069.2    1092.9    -529.6    1059.2       843 

Scaled residuals: 
     Min       1Q   Median       3Q      Max 
-2.37514 -0.87719  0.04037  0.82164  2.67698 

Random effects:
 Groups     Name        Variance  Std.Dev. 
 subject_id (Intercept) 1.538e-16 1.240e-08
 model_name (Intercept) 5.699e-01 7.549e-01
Number of obs: 848, groups:  subject_id, 37; model_name, 6

Fixed effects:
                    Estimate Std. Error z value Pr(>|z|)    
(Intercept)         6.342349   1.539067   4.121 3.77e-05 ***
N100                0.023113   0.004639   4.982 6.28e-07 ***
log(fixation_time) -0.798408   0.162187  -4.923 8.53e-07 ***
---
Signif. codes:  0 '***' 0.001 '**' 0.01 '*' 0.05 '.'

boundary (singular) fit: see help('isSingular')


In [17]:
%%R -i df
dt <- data.table(df)

m3_reg <- lm(mt ~ N100 + log(fixation_time), data = dt)

print("R-squared: ")
print(summary(m3_reg)$r.squared)
print("Adjusted R-squared: ")
print(summary(m3_reg)$adj.r.squared)
summary(m3_reg)

[1] "R-squared: "
[1] 0.06381298
[1] "Adjusted R-squared: "
[1] 0.06188468

Call:
lm(formula = mt ~ N100 + log(fixation_time), data = dt)

Residuals:
     Min       1Q   Median       3Q      Max 
-1.10298 -0.45408 -0.00101  0.45222  0.77535 

Coefficients:
                     Estimate Std. Error t value Pr(>|t|)    
(Intercept)         2.8376035  0.2763266  10.269  < 2e-16 ***
N100                0.0058803  0.0008135   7.229 9.88e-13 ***
log(fixation_time) -0.1671465  0.0293523  -5.694 1.64e-08 ***
---
Signif. codes:  0 '***' 0.001 '**' 0.01 '*' 0.05 '.' 0.1 ' ' 1

Residual standard error: 0.4521 on 971 degrees of freedom
Multiple R-squared:  0.06381,	Adjusted R-squared:  0.06188 
F-statistic: 33.09 on 2 and 971 DF,  p-value: 1.249e-14



### M_eye:

In [18]:
%%R -i df
dt <- data.table(df)

dt$mt = (dt$mt - min(dt$mt)) / (max(dt$mt) - min(dt$mt))
table(dt$mt)
#   0 0.5   1
# 424 126 424

# ignore the 0.5 group
dt <- dt[mt != 0.5]

m_comb <- glmer(mt ~ N100 + log(pixel_speed) + log(fixation_time) + (1|model_name) + (1|subject_id), data = dt, family = binomial(link = "logit"))

print("AIC:")
print(AIC(m_comb))
print("BIC:")
print(BIC(m_comb))
summary(m_comb)

[1] "AIC:"
[1] 1040.261
[1] "BIC:"
[1] 1068.718
Generalized linear mixed model fit by maximum likelihood (Laplace
  Approximation) [glmerMod]
 Family: binomial  ( logit )
Formula: 
mt ~ N100 + log(pixel_speed) + log(fixation_time) + (1 | model_name) +  
    (1 | subject_id)
   Data: dt

      AIC       BIC    logLik -2*log(L)  df.resid 
   1040.3    1068.7    -514.1    1028.3       842 

Scaled residuals: 
     Min       1Q   Median       3Q      Max 
-3.05871 -0.83356 -0.00374  0.84242  2.97718 

Random effects:
 Groups     Name        Variance Std.Dev.
 subject_id (Intercept) 0.0000   0.000   
 model_name (Intercept) 0.5551   0.745   
Number of obs: 848, groups:  subject_id, 37; model_name, 6

Fixed effects:
                    Estimate Std. Error z value Pr(>|z|)    
(Intercept)         7.682148   1.637121   4.692 2.70e-06 ***
N100                0.010470   0.005199   2.014    0.044 *  
log(pixel_speed)    0.961743   0.179468   5.359 8.37e-08 ***
log(fixation_time) -0.921591   0.173

boundary (singular) fit: see help('isSingular')


In [19]:
%%R -i df
dt <- data.table(df)

m_eye_reg <- lm(mt ~ N100 + log(pixel_speed) + log(fixation_time), data = dt)
print("R-squared: ")
print(summary(m_eye_reg)$r.squared)
print("Adjusted R-squared: ")
print(summary(m_eye_reg)$adj.r.squared)
summary(m_eye_reg)

[1] "R-squared: "
[1] 0.09923895
[1] "Adjusted R-squared: "
[1] 0.09645309

Call:
lm(formula = mt ~ N100 + log(pixel_speed) + log(fixation_time), 
    data = dt)

Residuals:
     Min       1Q   Median       3Q      Max 
-0.93795 -0.45737  0.00659  0.46791  0.87099 

Coefficients:
                     Estimate Std. Error t value Pr(>|t|)    
(Intercept)         2.9280320  0.2715826  10.781  < 2e-16 ***
N100                0.0027730  0.0009437   2.939  0.00338 ** 
log(pixel_speed)    0.1900601  0.0307715   6.176 9.63e-10 ***
log(fixation_time) -0.1706621  0.0288120  -5.923 4.38e-09 ***
---
Signif. codes:  0 '***' 0.001 '**' 0.01 '*' 0.05 '.' 0.1 ' ' 1

Residual standard error: 0.4437 on 970 degrees of freedom
Multiple R-squared:  0.09924,	Adjusted R-squared:  0.09645 
F-statistic: 35.62 on 3 and 970 DF,  p-value: < 2.2e-16



### NLL + Eye

In [20]:
%%R -i df
dt <- data.table(df)

dt$mt = (dt$mt - min(dt$mt)) / (max(dt$mt) - min(dt$mt))
table(dt$mt)
#   0 0.5   1
# 424 126 424

# ignore the 0.5 group
dt <- dt[mt != 0.5]

m_comb <- glmer(mt ~ nll + N100 + log(pixel_speed) + log(fixation_time) + (1|model_name) + (1|subject_id), data = dt, family = binomial(link = "logit"))

print("AIC:")
print(AIC(m_comb))
print("BIC:")
print(BIC(m_comb))
summary(m_comb)

[1] "AIC:"
[1] 982.7336
[1] "BIC:"
[1] 1015.934
Generalized linear mixed model fit by maximum likelihood (Laplace
  Approximation) [glmerMod]
 Family: binomial  ( logit )
Formula: mt ~ nll + N100 + log(pixel_speed) + log(fixation_time) + (1 |  
    model_name) + (1 | subject_id)
   Data: dt

      AIC       BIC    logLik -2*log(L)  df.resid 
    982.7    1015.9    -484.4     968.7       841 

Scaled residuals: 
    Min      1Q  Median      3Q     Max 
-8.1547 -0.7599 -0.0495  0.8036  3.9141 

Random effects:
 Groups     Name        Variance Std.Dev.
 subject_id (Intercept) 0.000    0.000   
 model_name (Intercept) 1.128    1.062   
Number of obs: 848, groups:  subject_id, 37; model_name, 6

Fixed effects:
                    Estimate Std. Error z value Pr(>|z|)    
(Intercept)         3.035481   1.878184   1.616    0.106    
nll                 0.023542   0.003494   6.738 1.60e-11 ***
N100               -0.001472   0.005618  -0.262    0.793    
log(pixel_speed)    0.783412   0.193544  

boundary (singular) fit: see help('isSingular')


In [21]:
%%R -i df
dt <- data.table(df)

m_nll_eye_reg <- lm(mt ~ nll + log(fixation_time) + log(pixel_speed) + N100, data = dt)
print("R-squared: ")
print(summary(m_nll_eye_reg)$r.squared)
print("Adjusted R-squared: ")
print(summary(m_nll_eye_reg)$adj.r.squared)
summary(m_nll_eye_reg)

[1] "R-squared: "
[1] 0.119091
[1] "Adjusted R-squared: "
[1] 0.1154547

Call:
lm(formula = mt ~ nll + log(fixation_time) + log(pixel_speed) + 
    N100, data = dt)

Residuals:
     Min       1Q   Median       3Q      Max 
-1.09194 -0.43797 -0.00204  0.46480  0.86349 

Coefficients:
                     Estimate Std. Error t value Pr(>|t|)    
(Intercept)         2.4974512  0.2840705   8.792  < 2e-16 ***
nll                 0.0015187  0.0003250   4.673 3.39e-06 ***
log(fixation_time) -0.1438883  0.0290775  -4.948 8.82e-07 ***
log(pixel_speed)    0.1641222  0.0309481   5.303 1.41e-07 ***
N100                0.0017996  0.0009566   1.881   0.0602 .  
---
Signif. codes:  0 '***' 0.001 '**' 0.01 '*' 0.05 '.' 0.1 ' ' 1

Residual standard error: 0.439 on 969 degrees of freedom
Multiple R-squared:  0.1191,	Adjusted R-squared:  0.1155 
F-statistic: 32.75 on 4 and 969 DF,  p-value: < 2.2e-16



### random effects

In [22]:
%%R
dt <- data.table(df)

dt$mt = (dt$mt - min(dt$mt)) / (max(dt$mt) - min(dt$mt))
table(dt$mt)

# ignore the 0.5 group
dt <- dt[mt != 0.5]

m4 <- glmer(mt ~ nll + exp + N100 + pixel_speed + (1|model_name), data = dt, family = binomial(link = "logit"))
summary(m4)

# dt_sub1 <- dt[model_name == "gpt-4"]
# dt_sub2 <- ...

Generalized linear mixed model fit by maximum likelihood (Laplace
  Approximation) [glmerMod]
 Family: binomial  ( logit )
Formula: mt ~ nll + exp + N100 + pixel_speed + (1 | model_name)
   Data: dt

      AIC       BIC    logLik -2*log(L)  df.resid 
    979.2    1007.7    -483.6     967.2       842 

Scaled residuals: 
    Min      1Q  Median      3Q     Max 
-9.7194 -0.7410 -0.0527  0.7500  4.3325 

Random effects:
 Groups     Name        Variance Std.Dev.
 model_name (Intercept) 0.9804   0.9902  
Number of obs: 848, groups:  model_name, 6

Fixed effects:
             Estimate Std. Error z value Pr(>|z|)    
(Intercept) -5.751057   0.719527  -7.993 1.32e-15 ***
nll          0.022806   0.003425   6.659 2.76e-11 ***
exp          1.144212   0.256493   4.461 8.16e-06 ***
N100        -0.004087   0.005223  -0.783  0.43388    
pixel_speed  0.286314   0.099850   2.867  0.00414 ** 
---
Signif. codes:  0 '***' 0.001 '**' 0.01 '*' 0.05 '.' 0.1 ' ' 1

Correlation of Fixed Effects:
            (I